# Kafka Streaming ETL - Real-time with readStream

Continuously consumes events from Confluent Cloud Kafka using Spark Structured Streaming.
Implements Bronze -> Silver -> Gold medallion architecture.

In [ ]:
# Confluent Cloud Configuration
KAFKA_BOOTSTRAP_SERVERS = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"
KAFKA_API_KEY = "XDNCGI37HYWXKQ43"
KAFKA_API_SECRET = "cfltkC3TGvQcqDzS0S/se0r/QgHrzyxoZLXKS5JudeBGmnq9PmTpf+KzEYpZThdA"

# Topics
TOPIC_ORDERS_CREATED = "orders.created"
TOPIC_PAYMENTS = "payments.authorized"

# Lakehouse: lakehouse_biju_analytics
WORKSPACE_ID = "11782c23-baa6-4054-b6bb-122aa8f0e470"
LAKEHOUSE_ID = "cf71c3e4-7b3a-41ac-8dea-ec0bd6764f36"
ABFSS_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"

SCHEMA_PATH = "bijugeneric"
STREAMING_TIMEOUT_SECONDS = 60

print(f"Kafka: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"Lakehouse: lakehouse_biju_analytics/{SCHEMA_PATH}")
print(f"Streaming timeout: {STREAMING_TIMEOUT_SECONDS}s")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 13, Finished, Available, Finished)

Kafka: pkc-921jm.us-east-2.aws.confluent.cloud:9092
Lakehouse: lakehouse_biju_analytics/bijugeneric
Streaming timeout: 60s


In [ ]:
from pyspark.sql.functions import col, current_timestamp, get_json_object, sum, count, avg
from delta.tables import DeltaTable

# Kafka connection options
kafka_options = {
    "kafka.bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": f'org.apache.kafka.common.security.plain.PlainLoginModule required username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";',
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

# Delta table paths
BRONZE_ORDERS_STREAM = f"{ABFSS_ROOT}/Tables/{SCHEMA_PATH}/bronze_orders_stream"
BRONZE_PAYMENTS_STREAM = f"{ABFSS_ROOT}/Tables/{SCHEMA_PATH}/bronze_payments_stream"
SILVER_ORDERS_STREAM = f"{ABFSS_ROOT}/Tables/{SCHEMA_PATH}/silver_orders_stream"
GOLD_SUMMARY_STREAM = f"{ABFSS_ROOT}/Tables/{SCHEMA_PATH}/gold_sales_summary_stream"

# Checkpoints
CHECKPOINT_ORDERS = f"{ABFSS_ROOT}/Files/checkpoints/{SCHEMA_PATH}/orders_stream"
CHECKPOINT_PAYMENTS = f"{ABFSS_ROOT}/Files/checkpoints/{SCHEMA_PATH}/payments_stream"
CHECKPOINT_SILVER = f"{ABFSS_ROOT}/Files/checkpoints/{SCHEMA_PATH}/silver_stream"

print("Configuration ready")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 14, Finished, Available, Finished)

Configuration ready


## Bronze Layer - Raw data from Kafka

In [ ]:
# Orders stream - Bronze
df_orders_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", TOPIC_ORDERS_CREATED)
    .load()
    .withColumn("key_str", col("key").cast("string"))
    .withColumn("value_str", col("value").cast("string"))
    .withColumn("ingestion_time", current_timestamp())
    .select(
        col("key_str").alias("key"),
        col("value_str"),
        col("timestamp").alias("kafka_timestamp"),
        col("ingestion_time")
    )
)

print("Orders Bronze stream created")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 15, Finished, Available, Finished)

Orders Bronze stream created


In [ ]:
# Payments stream - Bronze
df_payments_stream = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .option("subscribe", TOPIC_PAYMENTS)
    .load()
    .withColumn("key_str", col("key").cast("string"))
    .withColumn("value_str", col("value").cast("string"))
    .withColumn("ingestion_time", current_timestamp())
    .select(
        col("key_str").alias("key"),
        col("value_str"),
        col("timestamp").alias("kafka_timestamp"),
        col("ingestion_time")
    )
)

print("Payments Bronze stream created")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 16, Finished, Available, Finished)

Payments Bronze stream created


## Silver Layer - Parsed and cleaned data using foreachBatch

In [ ]:
def process_orders_to_silver(batch_df, batch_id):
    """Process each micro-batch: write Bronze and transform to Silver"""
    if batch_df.count() == 0:
        return
    
    # Write to Bronze
    batch_df.write.format("delta").mode("append").save(BRONZE_ORDERS_STREAM)
    
    # Transform to Silver (parse JSON)
    df_silver = batch_df.select(
        get_json_object("value_str", "$.order_id").alias("order_id"),
        get_json_object("value_str", "$.customer_id").alias("customer_id"),
        get_json_object("value_str", "$.region").alias("region"),
        get_json_object("value_str", "$.total_amount").cast("double").alias("total_amount"),
        get_json_object("value_str", "$.event_time").alias("event_time"),
        col("kafka_timestamp"),
        col("ingestion_time")
    ).dropDuplicates(["order_id"])
    
    # Write to Silver (merge for deduplication)
    if DeltaTable.isDeltaTable(spark, SILVER_ORDERS_STREAM):
        delta_silver = DeltaTable.forPath(spark, SILVER_ORDERS_STREAM)
        delta_silver.alias("target").merge(
            df_silver.alias("source"),
            "target.order_id = source.order_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        df_silver.write.format("delta").mode("overwrite").save(SILVER_ORDERS_STREAM)
    
    print(f"Batch {batch_id}: {batch_df.count()} orders -> Bronze & Silver")

print("Silver processor defined")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 17, Finished, Available, Finished)

Silver processor defined


## Gold Layer - Aggregations updated with each batch

In [ ]:
def update_gold_summary():
    """Rebuild Gold summary from Silver"""
    try:
        df_silver = spark.read.format("delta").load(SILVER_ORDERS_STREAM)
        
        df_gold = df_silver.groupBy("region").agg(
            count("order_id").alias("total_orders"),
            sum("total_amount").alias("total_revenue"),
            avg("total_amount").alias("avg_order_value")
        ).orderBy(col("total_revenue").desc())
        
        df_gold.write.format("delta").mode("overwrite").save(GOLD_SUMMARY_STREAM)
        return df_gold.count()
    except Exception as e:
        print(f"Gold update skipped: {e}")
        return 0

print("Gold updater defined")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 18, Finished, Available, Finished)

Gold updater defined


## Start Streaming

In [ ]:
print("Starting streaming queries...")

# Orders stream with Bronze -> Silver processing
query_orders = (
    df_orders_stream.writeStream
    .foreachBatch(process_orders_to_silver)
    .option("checkpointLocation", CHECKPOINT_ORDERS)
    .start()
)
print("Orders stream started (Bronze -> Silver)")

# Payments stream (Bronze only)
query_payments = (
    df_payments_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PAYMENTS)
    .start(BRONZE_PAYMENTS_STREAM)
)
print("Payments stream started (Bronze)")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 19, Finished, Available, Finished)

Starting streaming queries...
Orders stream started (Bronze -> Silver)
Payments stream started (Bronze)


In [ ]:
import time

print(f"\nStreaming for {STREAMING_TIMEOUT_SECONDS} seconds...")
print("Updating Gold layer every 20 seconds")
print("")

start_time = time.time()
last_gold_update = 0

while time.time() - start_time < STREAMING_TIMEOUT_SECONDS:
    elapsed = int(time.time() - start_time)
    
    # Update Gold every 20 seconds
    if elapsed - last_gold_update >= 20:
        gold_regions = update_gold_summary()
        last_gold_update = elapsed
        print(f"[{elapsed}s] Gold updated: {gold_regions} regions")
    else:
        print(f"[{elapsed}s] Streaming...")
    
    time.sleep(10)

print("\nStopping streams...")
query_orders.stop()
query_payments.stop()
print("Stopped")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 20, Finished, Available, Finished)


Streaming for 60 seconds...
Updating Gold layer every 20 seconds

[0s] Streaming...
[10s] Streaming...
[20s] Gold updated: 4 regions
[33s] Streaming...
[43s] Gold updated: 4 regions

Stopping streams...
Stopped


In [ ]:
# Final Gold update
print("Final Gold layer update...")
update_gold_summary()

print("")
print("=== Streaming ETL Complete ===")
print(f"Schema: {SCHEMA_PATH}")
print("")

tables = [
    ("bronze_orders_stream", BRONZE_ORDERS_STREAM),
    ("bronze_payments_stream", BRONZE_PAYMENTS_STREAM),
    ("silver_orders_stream", SILVER_ORDERS_STREAM),
    ("gold_sales_summary_stream", GOLD_SUMMARY_STREAM)
]

for name, path in tables:
    try:
        c = spark.read.format("delta").load(path).count()
        print(f"{name}: {c} records")
    except Exception as e:
        print(f"{name}: not created")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 21, Finished, Available, Finished)

Final Gold layer update...

=== Streaming ETL Complete ===
Schema: bijugeneric

bronze_orders_stream: 167 records
bronze_payments_stream: 87 records
silver_orders_stream: 95 records
gold_sales_summary_stream: 4 records


In [ ]:
# Show Gold summary
print("\n=== Gold Sales Summary ===")
try:
    spark.read.format("delta").load(GOLD_SUMMARY_STREAM).show()
except:
    print("No Gold data yet")

StatementMeta(, 06e07b00-cd3e-4e81-9f51-d0320a4d4bc2, 22, Finished, Available, Finished)


=== Gold Sales Summary ===
+-------+------------+------------------+------------------+
| region|total_orders|     total_revenue|   avg_order_value|
+-------+------------+------------------+------------------+
|EU-West|          26| 40759.07000000001|1567.6565384615387|
|US-East|          30|29189.209999999995| 972.9736666666665|
|US-West|          25|          28069.17|1122.7667999999999|
|   APAC|          14|13199.619999999999| 942.8299999999999|
+-------+------------+------------------+------------------+

